# SV model with normal-mixture approximation

本 Notebook 演示如何在本地数据文件夹中加载分钟级期货数据，使用混合正态近似的随机波动（SV）模型进行快速 MCMC 演示。所有注释使用中文，print 输出和图表元素使用英文，便于跨平台显示。

In [ ]:
# 导入依赖，注释使用中文
import numpy as np
import pandas as pd
from pathlib import Path

# 可视化与进度显示
import matplotlib.pyplot as plt
import seaborn as sns

# 自定义模块
from sv_toolkit.data import list_csv_files, load_dataset
from sv_toolkit.mcmc import run_mcmc_sv
from sv_toolkit.plotting import plot_returns, plot_volatility

# 设置 matplotlib 显示风格
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# 确定数据目录，默认为当前仓库下的 2005年__20250905 文件夹
data_dir = Path('2005年__20250905')
fig_dir = Path('figures')

print('Environment ready. Please adjust paths if needed.')

## 1. 浏览数据文件

使用自带的工具函数列出数据目录中的前若干个 CSV 文件名称，便于选择要处理的文件。

In [ ]:
# 列出数据目录中的前 5 个文件，避免一次性打印全部
preview_files = list_csv_files(data_dir, max_files=5)
print('Preview finished.')

## 2. 读取单个/少量文件并构造收益率

下方示例只读取第一个文件，并且可以通过 max_rows_per_file 控制读取的行数。例如数据量过大时，可以先读取前 10000 行进行调试。start_time / end_time 参数可用于限定日期范围。

In [ ]:
# 选择数据子集，便于快速跑通流程
r, y_star, df = load_dataset(
    data_dir,
    contract_code=None,  # 可以填入具体合约代码，比如 'A0505.XDCE'
    start_time=None,     # 可以填入 '2005-01-01' 这样的字符串
    end_time=None,       # 可以填入 '2008-12-31' 等字符串
    max_files=1,         # 只读取一个文件进行示范
    max_rows_per_file=10000,  # 限制行数，方便快速运行
)

print(f'Data shape after processing: returns={len(r)}, dataframe={len(df)} rows')

## 3. 运行简化版 MCMC

为了演示，本示例使用较少的迭代次数，并设置 progress_every 参数让运行过程中定期打印进度。实际研究时可适当增加迭代次数与烧入期。

In [ ]:
# 运行一个轻量级的 MCMC 样例，确保进度信息可见
mcmc_results = run_mcmc_sv(
    r=r,
    y_star=y_star,
    n_iter=120,      # 演示用较小迭代次数
    burn_in=40,
    thin=2,
    rng_seed=2025,
    progress_every=20,
)

print('Sampling finished. Posterior samples available in mcmc_results dict.')

## 4. 绘制收益率与隐含波动率

图表标题、坐标轴与图例使用英文；图片将会保存到 figures 目录，格式为 PNG。

In [ ]:
# 绘制收益率轨迹并保存
returns_png = plot_returns(df, fig_dir, title_suffix='demo')
print(f'Returns figure saved to: {returns_png}')

# 绘制波动率轨迹：使用后验样本 h 计算
if len(mcmc_results['h']) > 0:
    vol_png = plot_volatility(mcmc_results['h'], df, fig_dir, title_suffix='demo')
    print(f'Volatility figure saved to: {vol_png}')
else:
    print('No h samples available to plot volatility.')

# 在 Notebook 中直接展示最近生成的图片
from IPython.display import Image, display
display(Image(filename=returns_png))
if len(mcmc_results['h']) > 0:
    display(Image(filename=vol_png))